# 4. Fine-Tuning Data Preparation

This notebook prepares a dataset for instruction tuning (SFT) of a
language model. We convert the structured advice database into the
conversational JSONL format required by the Mistral fine-tuning API.

### Objectives
1. Load the curated advice database from CSV.
2. Augment data by generating multiple user query variations per emotion.
3. Format into JSONL with system/user/assistant roles.
4. Validate the output file.

In [40]:
import pandas as pd
import json
import os

# --- Configuration ---
# Paths are relative to the 'notebooks' folder
INPUT_CSV_PATH = "../data/conseils_emotions.csv"
OUTPUT_JSONL_PATH = "../data/fine_tuning_data.jsonl"

print(f"[INFO] Input file: {INPUT_CSV_PATH}")
print(f"[INFO] Output file: {OUTPUT_JSONL_PATH}")

[INFO] Input file: ../data/conseils_emotions.csv
[INFO] Output file: ../data/fine_tuning_data.jsonl


### 1. Data Loading
We load the therapeutic advice database. This file contains the expert knowledge we want the model to memorize and adopt as its tone of voice.

In [41]:
if not os.path.exists(INPUT_CSV_PATH):
    print(f"[ERROR] Input file not found at {INPUT_CSV_PATH}")
else:
    df = pd.read_csv(INPUT_CSV_PATH)
    print(f"[SUCCESS] CSV loaded. Rows found: {len(df)}")
    # Display the first few rows to verify structure
    print(df.head())

[SUCCESS] CSV loaded. Rows found: 6
   emotion                                             advice  \
0  sadness  I hear that you're going through a tough time....   
1      joy  That is wonderful to hear! Hold onto this feel...   
2     fear  It sounds like you are feeling anxious or scar...   
3    anger  I sense a lot of frustration. Anger is often a...   
4     love  That is a beautiful emotion. Love and connecti...   

                                               notes  
0  Ask the user if they want to talk about what t...  
1  Encourage the user to journal this moment to r...  
2  Propose a grounding exercise (naming 5 objects...  
3  Ask if there is a specific event that caused t...  
4   Suggest sending a nice message to the loved one.  


### 2. Formatting for Instruction Tuning
To fine-tune a chat model, we need to provide examples of valid conversations. Each example must follow a specific schema:
* **System:** Defines the behavior of the AI (e.g., "You are an empathetic assistant").
* **User:** The simulated input from a human.
* **Assistant:** The ideal response we expect from the AI.

**Data Augmentation Strategy:**
Since the dataset is small, we synthetically increase its size by creating multiple variations of the User prompt for each advice entry (e.g., "I feel sad", "Why am I sad?", "Feeling down").

In [42]:
training_data = []

# Templates to simulate different ways a user might express an emotion
user_templates = [
    "I feel {emotion}",
    "I am feeling very {emotion} today",
    "I am overwhelmed by {emotion}, what should I do?",
    "Can you help me with my {emotion}?"
]

print("[INFO] Starting data conversion and augmentation...")

for index, row in df.iterrows():
    # Normalize emotion text
    emotion = str(row['emotion']).lower().strip()
    advice = str(row['advice'])
    note = str(row['notes'])
    
    # Construct the ideal Assistant response
    # We combine the direct advice with the psychological note for a comprehensive answer
    assistant_response = f"{advice} {note}"

    # Generate multiple training examples for this single row
    for template in user_templates:
        user_message = template.format(emotion=emotion)
        
        # Standard Mistral/OpenAI JSONL structure
        entry = {
            "messages": [
                {"role": "system", "content": "You are MindCare, an empathetic mental health assistant."},
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": assistant_response}
            ]
        }
        training_data.append(entry)

print(f"[SUCCESS] Conversion complete.")
print(f"[METRICS] Original rows: {len(df)} -> Generated training examples: {len(training_data)}")

[INFO] Starting data conversion and augmentation...
[SUCCESS] Conversion complete.
[METRICS] Original rows: 6 -> Generated training examples: 24


### 3. Export and Validation
We save the dataset in **JSON Lines (.jsonl)** format, where each line is a valid JSON object representing one conversation. This is the standard format for uploading data to fine-tuning platforms.

In [43]:
try:
    with open(OUTPUT_JSONL_PATH, 'w', encoding='utf-8') as f:
        for entry in training_data:
            json.dump(entry, f)
            f.write('\n')
            
    print(f"[SUCCESS] Training data saved to '{OUTPUT_JSONL_PATH}'")
    
    # Validation: Print the first entry to check format
    print("\n--- Sample Entry (Formatted) ---")
    print(json.dumps(training_data[0], indent=2))
    print("-------------------------------")
    
except Exception as e:
    print(f"[ERROR] Failed to write JSONL file: {e}")

[SUCCESS] Training data saved to '../data/fine_tuning_data.jsonl'

--- Sample Entry (Formatted) ---
{
  "messages": [
    {
      "role": "system",
      "content": "You are MindCare, an empathetic mental health assistant."
    },
    {
      "role": "user",
      "content": "I feel sadness"
    },
    {
      "role": "assistant",
      "content": "I hear that you're going through a tough time. It's important to let yourself feel these emotions rather than suppressing them. Try to do one small thing today that brings you a tiny bit of comfort, like a hot tea, a short walk, or calling a friend. Remember, this feeling is temporary. Ask the user if they want to talk about what triggered this feeling."
    }
  ]
}
-------------------------------
